# 14. Genome-wide selection scans

This notebook documents the `Ag3` methods for running **genome-wide selection
scans** (GWSS): sliding-window statistics computed along a chromosome arm
that detect regions of unusually low haplotype or genotype diversity, a
signature of a recent selective sweep (e.g. driven by insecticide
resistance). Four families of statistic are covered:

- **H12** — haplotype homozygosity within a single cohort (needs phased data).
- **G123** — the diplotype (genotype) analogue of H12, computed on unphased
  data.
- **H1X** — like H12 but measures haplotype diversity *shared between two
  cohorts*, for detecting shared/introgressed sweeps.
- **iHS** — within-cohort extended haplotype homozygosity (EHH) decay
  asymmetry between ancestral and derived alleles, sensitive to ongoing,
  not-yet-fixed sweeps.
- **XP-EHH** — EHH decay compared *between two cohorts*, sensitive to sweeps
  that are complete (or near-complete) in one cohort by using the other,
  still-polymorphic cohort as a diversity reference.

Each statistic has a plain computation method (returns arrays) and one or
more `plot_*` methods built on top of it, which additionally render a bokeh
figure with a genome position track and a genes track underneath (linked
x-axis panning/zooming). H12, G123 and H1X all have `*_calibration` /
`plot_*_calibration` companions, used to choose a sensible window size
before running the full scan.

**Diagram opportunity:** A diagram of a chromosome region undergoing a
selective sweep (allele frequency rising towards fixation, haplotype
diversity collapsing around the causal variant), with small panels showing
how H12 (single-cohort haplotype homozygosity spike), G123 (single-cohort
diplotype homozygosity spike), H1X (shared haplotype spike between two
cohorts), iHS (EHH decay asymmetry between alleles) and XP-EHH (EHH decay
difference between cohorts) each register the same event — plus a small
"which statistic when" comparison table (single- vs two-cohort, phased vs
unphased, ongoing vs completed sweep). Would fit right after this
paragraph.

To keep runtime reasonable while still executing real, live queries against
the data, examples below use small cohorts (~10-20 samples, fixed via
`cohort_size`/`max_cohort_size`) and either a single chromosome arm or the
X contig, rather than the full release cohorts across all chromosome arms.


In [1]:
import malariagen_data
ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../../gcs_cache"),
    results_cache="../../results_cache",
)
ag3


/opt/homebrew/Caskroom/miniconda/base/envs/malariagen2/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


<MalariaGEN Ag3 API client>
Storage URL                           : simplecache::gs://vo_agam_release_master_us_central1
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
Results cache                         : /Users/katie.barr/malariagen-data-python/results_cache
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.8.0.post13+b769b728
Client location                       : England, United Kingdom
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.8.0.post13+b769b728/Ag3.html

We'll reuse a couple of small, well-separated cohorts throughout the H12,
H1X and iHS/XP-EHH examples: two Malian *An. coluzzii* / *An. gambiae*
cohorts from the same admin2 area and year (for within- and
between-cohort comparisons), and, for XP-EHH, a geographically distant
Mayotte cohort as the second comparison population.


In [2]:
coh1_query = "cohort_admin2_year == 'ML-2_Kati_colu_2014'"
coh2_query = "cohort_admin2_year == 'ML-2_Kati_gamb_2014'"
mayotte_query = "country == 'Mayotte'"


## `h12_calibration`

Computes Garud's H12 (the summed squared frequency of the two most common
haplotypes, plus the summed squared frequencies of all others) in
consecutive windows of a fixed number of SNPs, repeated across several
candidate window sizes, for a single cohort of **phased** haplotypes. Used
to pick a window size before running `h12_gwss`: too small and per-window
H12 is noisy even with no sweep; too large and a real sweep's signal gets
diluted by the surrounding non-swept haplotypes in the same window. Returns
a dict mapping each window size (as a string) to an array of H12 values,
one per window, computed genome-wide across the whole contig at that window
size.

Parameters:
- **contig**: chromosome arm to analyse; phased haplotype data must exist
  for it.
- **analysis**: which phasing analysis/haplotype panel supplies the phased
  genotypes (e.g. `"gamb_colu"` restricts to the *gambiae*/*coluzzii*
  phasing panel) — changes both which samples are eligible and which SNPs
  were ascertained as the phasing panel's SNP grid.
- **sample_query** / **sample_query_options**: pandas query string (and
  optional extra `query()`/`eval()` kwargs) selecting which samples form
  the cohort.
- **sample_sets**: which release(s)/sample set(s) to draw samples from.
- **cohort_size**: if given, randomly down-sample the cohort to exactly
  this many samples (error if fewer are available) — fixes cohort size so
  H12 values are comparable across different cohorts.
- **min_cohort_size** / **max_cohort_size**: alternative to `cohort_size` —
  enforce a floor (error below it) and/or ceiling (down-sample above it)
  without forcing an exact size.
- **window_sizes**: sequence of window sizes (number of SNPs) to try;
  defaults span 100 to 20,000 SNPs.
- **random_seed**: seed controlling which samples get picked when
  down-sampling, for reproducibility.
- **chunks** / **inline_array**: low-level dask chunking controls for how
  the underlying zarr arrays are loaded; rarely changed from the defaults.


In [3]:
calibration_runs = ag3.h12_calibration(
    contig="3L",
    analysis="gamb_colu",
    sample_query=coh1_query,
    sample_sets="3.0",
    cohort_size=20,
)
{window_size: values.shape for window_size, values in calibration_runs.items()}


{'100': (113540,),
 '1000': (11354,),
 '10000': (1135,),
 '200': (56770,),
 '2000': (5677,),
 '20000': (567,),
 '500': (22708,),
 '5000': (2270,)}

## `plot_h12_calibration`

Runs `h12_calibration` and plots the resulting distribution of H12 values
(median line, with 25-75% and 5-95% percentile bands) against window size
on a log-scaled x-axis. Used to visually choose a window size where the
baseline H12 is reasonably low and tight (indicating enough SNPs per window
to average out noise) before committing to it for the full genome scan.

Parameters (in addition to all of `h12_calibration`'s): **title** (plot
title; defaults to the `sample_query` text) and **show** (if `True`,
display the plot inline; if `False`, return the bokeh `Figure` object
instead of displaying it).


In [4]:
ag3.plot_h12_calibration(
    contig="3L",
    analysis="gamb_colu",
    sample_query=coh1_query,
    sample_sets="3.0",
    cohort_size=20,
)


figure(id='p1009', ...)

## `h12_gwss`

Runs the H12 genome-wide selection scan for a single cohort: computes H12
in consecutive, non-overlapping windows of `window_size` SNPs across the
whole contig, guided by the calibration above. Returns three arrays: the
genomic midpoint position of each window, the H12 value for each window,
and the contig each window belongs to (useful mainly when a multi-contig
region such as `"2RL"` is passed).

Parameters:
- **window_size**: fixed number of SNPs per window (chosen via
  calibration); larger windows smooth the signal and reduce false
  positives but blur the exact peak location and lower its height.
- the remaining parameters (`analysis`, `sample_query`,
  `sample_query_options`, `sample_sets`, `cohort_size`, `min_cohort_size`,
  `max_cohort_size`, `random_seed`, `chunks`, `inline_array`) have the same
  meaning as in `h12_calibration`.


In [5]:
import pandas as pd

x, h12, contigs = ag3.h12_gwss(
    contig="3L",
    analysis="gamb_colu",
    window_size=2_000,
    sample_query=coh1_query,
    sample_sets="3.0",
    cohort_size=20,
)
df_h12 = pd.DataFrame({"position": x, "h12": h12, "contig_index": contigs})
df_h12.sort_values("h12", ascending=False).head()


,position,h12,contig_index
1069,1.165186e+07,0.43125,3
1023,1.123218e+07,0.34500,3
1070,1.165752e+07,0.32125,3
1018,1.113732e+07,0.29625,3
1020,1.118474e+07,0.28875,3


## `plot_h12_gwss`

Convenience wrapper that runs `h12_gwss` and produces a two-panel bokeh
figure: the top panel scatter-plots H12 against genomic position (points
coloured by contig, relevant only for multi-contig regions), and the
bottom panel shows gene models over the same x-range, with linked
panning/zooming between the two.

Parameters beyond `h12_gwss`: **title**; **sizing_mode**, **width**,
**track_height**, **genes_height** (layout sizing of the combined figure
and its two panels); **contig_colors** (colour assigned to each contig,
only visible with a multi-contig region); **show**; **output_backend**
(bokeh rendering backend — `"webgl"`, the default, handles the many points
in a scan faster than `"canvas"` or `"svg"`); **gene_labels** /
**gene_labelset** (custom text labels, or a full custom `LabelSet`, for
genes in the bottom track).


In [6]:
ag3.plot_h12_gwss(
    contig="3L",
    analysis="gamb_colu",
    window_size=2_000,
    sample_query=coh1_query,
    sample_sets="3.0",
    cohort_size=20,
)


Load genome features: ⠋ (0:00:00.00)

Load genome features: ⠙ (0:00:00.10)

Load genome features: ⠹ (0:00:00.20)

GridPlot(id='p1184', ...)

## `plot_h12_gwss_multi_panel`

Runs the H12 GWSS separately for several cohorts and stacks the resulting
tracks vertically, one panel per cohort with a linked x-axis, above a
single shared genes track — useful for comparing at a glance where sweeps
occur across multiple populations.

Parameters beyond `plot_h12_gwss`:
- **cohorts**: either the name of a predefined cohort grouping (e.g.
  `"admin2_year"`, which automatically splits the queried samples into one
  cohort per distinct admin2 + year combination), or an explicit dict
  mapping cohort labels to their own `sample_query` strings, as used
  below.
- **window_size**: accepts either a single int (used for every cohort) or
  a dict mapping cohort label to its own window size, so different
  cohorts can use different resolutions.
- **track_height** here sets the height of *each* per-cohort panel, not
  just a single track.


In [7]:
cohorts = {"colu": coh1_query, "gamb": coh2_query}

ag3.plot_h12_gwss_multi_panel(
    contig="3L",
    analysis="gamb_colu",
    cohorts=cohorts,
    window_size=2_000,
    sample_sets="3.0",
    cohort_size=20,
)


Load sample metadata: ⠋ (0:00:00.00)

Load sample metadata: ⠙ (0:00:00.11)

Load sample metadata: ⠹ (0:00:00.20)

Load genome features: ⠋ (0:00:00.00)

GridPlot(id='p1334', ...)

## `plot_h12_gwss_multi_overlay`

Like `plot_h12_gwss_multi_panel`, but overlays all cohorts' H12 tracks on
one shared set of axes with a legend (click a legend entry to toggle that
cohort's visibility), rather than stacking separate panels — better for
directly comparing peak height and location when there are only a few
cohorts.

Parameters beyond `plot_h12_gwss_multi_panel`: **colors** (sequence of
colours, one per cohort, cycling through bokeh's `Category10` palette by
default).


In [8]:
ag3.plot_h12_gwss_multi_overlay(
    contig="3L",
    analysis="gamb_colu",
    cohorts=cohorts,
    window_size=2_000,
    sample_sets="3.0",
    cohort_size=20,
)


Load genome features: ⠋ (0:00:00.00)

GridPlot(id='p1455', ...)

## `h1x_gwss`

Runs the H1X genome-wide scan, which sums the *joint* haplotype
frequencies between two cohorts in each window — i.e. how much haplotype
diversity is *shared* between them. A high H1X in a region means the same
haplotype(s) are common in both cohorts there, evidence of a shared (e.g.
introgressed) sweep, as opposed to two cohorts independently sweeping
different haplotypes at the same locus (which H1X would not flag, even
though H12 might be high in each cohort separately).

**Diagram opportunity:** A side-by-side diagram contrasting a "shared
sweep" (same haplotype swept in both cohorts, high H1X) against
"independent sweeps" (different haplotypes swept in each cohort, high H12
in each cohort individually but low H1X) — would make the distinction from
H12 concrete. Fits well right here, before the parameter list.

Parameters:
- **window_size**: same meaning as in H12 (number of SNPs per window).
- **cohort1_query** / **cohort2_query**: pandas queries defining the two
  cohorts being compared — required, with no default.
- **sample_query_options**, **analysis**, **sample_sets**, **cohort_size**,
  **min_cohort_size**, **max_cohort_size**, **random_seed**, **chunks**,
  **inline_array**: as in `h12_gwss`, applied identically when building
  each of the two cohorts' haplotype data.


In [9]:
x, h1x, contigs = ag3.h1x_gwss(
    contig="3L",
    analysis="gamb_colu",
    window_size=2_000,
    cohort1_query=coh1_query,
    cohort2_query=coh2_query,
    sample_sets="3.0",
    cohort_size=20,
)
df_h1x = pd.DataFrame({"position": x, "h1x": h1x, "contig_index": contigs})
df_h1x.sort_values("h1x", ascending=False).head()


,position,h1x,contig_index
5676,4.195776e+07,0.118125,3
5675,4.194945e+07,0.076875,3
5673,4.193131e+07,0.047500,3
5668,4.188903e+07,0.026875,3
5664,4.185587e+07,0.016250,3


## `plot_h1x_gwss`

Runs `h1x_gwss` and produces the same two-panel (H1X track + genes track)
bokeh figure style as `plot_h12_gwss`.

Parameters: the same layout parameters as `plot_h12_gwss` (**title**,
**sizing_mode**, **width**, **track_height**, **contig_colors**,
**genes_height**, **show**, **output_backend**, **gene_labels**,
**gene_labelset**), applied to the H1X track, plus `h1x_gwss`'s
computation parameters (**cohort1_query**, **cohort2_query**, etc.).


In [10]:
ag3.plot_h1x_gwss(
    contig="3L",
    analysis="gamb_colu",
    window_size=2_000,
    cohort1_query=coh1_query,
    cohort2_query=coh2_query,
    sample_sets="3.0",
    cohort_size=20,
)


Load genome features: ⠋ (0:00:00.00)

GridPlot(id='p1565', ...)

## `g123_calibration`

Computes Garud's G123 — the diplotype (genotype) analogue of H12: the
summed squared frequency of the three most common genotype "diplotypes"
in a window, plus the summed squared frequencies of the rest — across
several candidate window sizes, for calibration before running
`g123_gwss`. Because G123 works on **unphased** genotype calls, it can be
computed for taxa or cohorts that don't have a phasing analysis available,
unlike H12.

**Diagram opportunity:** A comparison diagram/table of G123 vs H12: same
underlying idea (homozygosity of the most common "type" in a window) but
G123 operates on unphased diplotypes (any sample, any taxon) while H12
needs phased haplotypes (only samples/taxa covered by a phasing analysis)
— would help explain why G123 exists alongside H12. Fits here, before the
parameter list.

Parameters:
- **contig**.
- **sites**: which SNPs form the analysis grid — `"segregating"`
  restricts to sites polymorphic within the selected cohort itself (a
  sweep reduces segregating-site density there, so windows can stretch
  further across the swept region and dilute the signal); `"all"` uses
  every site passing the site filters; or the identifier of a phasing
  analysis panel (e.g. `"gamb_colu"`), which uses that panel's SNP
  ascertainment as a fixed external site grid — an approximation to
  genome-wide segregating sites across the whole species complex, which
  avoids the window-stretching problem.
- **site_mask**: which site-filter mask to apply when restricting to
  high-quality sites.
- **sample_query** / **sample_query_options** / **sample_sets**: cohort
  selection, as elsewhere.
- **min_cohort_size** / **max_cohort_size**: cohort size floor/ceiling.
- **window_sizes**: candidate window sizes (numbers of sites) to
  calibrate over.
- **random_seed**, **inline_array**, **chunks**: as elsewhere.


In [11]:
g123_calibration_runs = ag3.g123_calibration(
    contig="3L",
    sites="gamb_colu",
    site_mask="gamb_colu",
    sample_sets="AG1000G-BF-A",
    sample_query='taxon == "gambiae"',
)
{window_size: values.shape for window_size, values in g123_calibration_runs.items()}


{'100': (113540,),
 '1000': (11354,),
 '200': (56770,),
 '2000': (5677,),
 '500': (22708,),
 '5000': (2270,)}

## `plot_g123_calibration`

Same calibration-curve plot as `plot_h12_calibration` (median line with
25-75% and 5-95% bands vs. window size, log x-axis), but for G123.

Parameters: **title**, **show**, plus all of `g123_calibration`'s
parameters — note that unlike `g123_gwss`, `sites` has no default here and
must be supplied explicitly.


In [12]:
ag3.plot_g123_calibration(
    contig="3L",
    sites="gamb_colu",
    site_mask="gamb_colu",
    sample_sets="AG1000G-BF-A",
    sample_query='taxon == "gambiae"',
)


figure(id='p1579', ...)

## `g123_gwss`

Runs the G123 genome-wide scan at a single, fixed `window_size`, returning
the window midpoint genomic positions and the G123 value for each window.

Parameters: **window_size** (number of sites per window), plus all of
`g123_calibration`'s cohort/site-selection parameters (minus
`window_sizes`). Below we use `sites="segregating"` to illustrate the
alternative to the phasing-panel-based `sites` used for calibration above.


In [13]:
x, g123 = ag3.g123_gwss(
    contig="3L",
    sites="segregating",
    site_mask="gamb_colu",
    window_size=1_000,
    sample_sets="AG1000G-BF-A",
    sample_query='taxon == "gambiae"',
    max_cohort_size=20,
)
df_g123 = pd.DataFrame({"position": x, "g123": g123})
df_g123.sort_values("g123", ascending=False).head()


,position,g123
0,1.009254e+05,0.065
1835,3.122209e+07,0.065
1827,3.113488e+07,0.065
1828,3.114368e+07,0.065
1829,3.115121e+07,0.065


## `plot_g123_gwss`

Runs `g123_gwss` and produces the two-panel (G123 track + genes track)
bokeh figure, in the same style as `plot_h12_gwss`.

Parameters: the same layout parameters as `plot_h12_gwss` (**title**,
**sizing_mode**, **width**, **track_height**, **genes_height**, **show**,
**output_backend**, **gene_labels**, **gene_labelset**), plus
`g123_gwss`'s computation parameters. Below we use `sites="all"` (every
filter-passing site, rather than a segregating- or panel-based subset) to
illustrate the third `sites` option.


In [14]:
ag3.plot_g123_gwss(
    contig="3L",
    sites="all",
    site_mask="gamb_colu",
    window_size=1_000,
    sample_sets="AG1000G-BF-A",
    sample_query='taxon == "gambiae"',
    min_cohort_size=10,
    max_cohort_size=20,
)


Load genome features: ⠋ (0:00:00.00)

GridPlot(id='p1754', ...)

## `ihs_gwss`

Runs the iHS genome-wide scan for a **single** cohort. iHS compares
integrated haplotype homozygosity (how slowly EHH decays away from a focal
SNP) between the ancestral and derived alleles at each SNP; a large `|iHS|`
means one allele's haplotype background has anomalously low diversity
relative to the other, consistent with a recent, still-segregating (not
yet fixed) sweep at that site. This makes iHS best suited to *ongoing,
incomplete* sweeps within one population — complementary to H12/G123,
which instead detect sweeps via an overall drop in local diversity and so
work best once a sweep is well underway or complete.

**Diagram opportunity:** A diagram of EHH decay curves away from a focal
SNP, one curve per allele, showing the asymmetric decay that produces a
large iHS — and how that differs from the two-cohort EHH comparison used
by XP-EHH below. Would fit well right here.

Parameters:
- **contig**, **analysis**, **sample_sets**, **sample_query**,
  **sample_query_options**: cohort selection, as elsewhere (uses phased
  haplotypes, so needs a phasing analysis).
- **window_size**: number of SNPs to summarise per window via percentiles
  (if falsy/`None`, one raw iHS value is returned per SNP instead of
  windowed summaries).
- **percentiles**: which percentile(s) of `|iHS|` to report per window
  (e.g. `(50, 75, 100)` reports the median, 75th percentile and max) — one
  output column per percentile requested.
- **standardize**: if `True` (default), standardizes raw iHS scores within
  derived-allele-count bins, so scores are comparable across sites of
  different frequency — recommended, since raw iHS is confounded by allele
  frequency.
- **standardization_bins** / **standardization_n_bins**: explicit
  allele-count bin edges to standardize within, or (if bins aren't given)
  how many equal-width bins to split allele counts into automatically.
- **standardization_diagnostics**: if `True`, additionally plot diagnostic
  figures of the standardization procedure.
- **filter_min_maf**: minimum minor allele frequency below which SNPs are
  dropped before computing EHH — removes rare variants that give noisy EHH
  estimates.
- **compute_min_maf**: minimum MAF below which a site's iHS isn't computed
  at all.
- **min_ehh**: EHH threshold below which homozygosity decay is truncated
  when integrating — controls how far out from the focal SNP EHH is
  tracked.
- **max_gap**: largest physical gap (bp) between variants that EHH may span
  before a site's score is discarded — guards against SNP-sparse regions
  artificially inflating extended haplotype homozygosity.
- **gap_scale**: rescales the effective distance for any gap larger than
  this, dampening the influence of very large physical gaps.
- **include_edges**: if `True`, still report a score for SNPs near a
  contig edge even if EHH hasn't decayed below `min_ehh` before the data
  run out (otherwise these are dropped).
- **use_threads**: if `True`, run the underlying scikit-allel iHS
  computation multithreaded.
- **min_cohort_size** / **max_cohort_size**, **random_seed**, **chunks**,
  **inline_array**: cohort-size floor/ceiling, down-sampling seed, and
  dask chunking, as elsewhere.


In [15]:
x, ihs = ag3.ihs_gwss(
    contig="2L",
    analysis="gamb_colu",
    window_size=1_000,
    sample_query=coh1_query,
    sample_sets="3.0",
    min_cohort_size=10,
    max_cohort_size=20,
)
df_ihs = pd.DataFrame(ihs, columns=[f"p{p}" for p in (50, 75, 100)])
df_ihs.insert(0, "position", x)
df_ihs.sort_values("p100", ascending=False).head()


,position,p50,p75,p100
5,3.268454e+06,1.014762,3.102074,10.092376
223,2.547289e+07,2.226404,4.809294,9.800020
6,3.400254e+06,1.449431,3.937811,9.558245
226,2.576540e+07,1.367816,3.023118,8.651227
222,2.534582e+07,1.607529,3.076410,8.386016


## `plot_ihs_gwss`

Runs `ihs_gwss` and produces the two-panel (iHS scatter + genes) bokeh
figure. When `window_size` is set and multiple `percentiles` are
requested, one differently-shaded point series is plotted per percentile
(darkest = highest percentile = most extreme).

Parameters beyond `ihs_gwss`: **palette** (name of a bokeh sequential
colour palette used to shade the percentile series); **title**,
**sizing_mode**, **width**, **track_height**, **genes_height**, **show**,
**output_backend**, **gene_labels**, **gene_labelset** (plot layout, as in
`plot_h12_gwss`). Below we request four percentiles to show the multiple
shaded series.


In [16]:
ag3.plot_ihs_gwss(
    contig="2L",
    analysis="gamb_colu",
    window_size=1_000,
    percentiles=(50, 60, 90, 100),
    sample_query=coh1_query,
    sample_sets="3.0",
    max_cohort_size=10,
)


Load genome features: ⠋ (0:00:00.00)

/Users/katie.barr/malariagen-data-python/malariagen_data/anopheles.py:1128: UserWarning: found multiple competing values for 'toolbar.active_inspect' property; using the latest value
  fig = bokeh.layouts.gridplot(


GridPlot(id='p1887', ...)

## `xpehh_gwss`

Runs the XP-EHH ("cross-population EHH") genome-wide scan, comparing
extended haplotype homozygosity decay *between two cohorts* at each SNP,
rather than between alleles within one cohort as iHS does. A large
positive XP-EHH indicates longer-range haplotype homozygosity (a more
complete/recent sweep) in cohort 1 relative to cohort 2 at that site; a
large negative value indicates the opposite. This makes XP-EHH well suited
to sweeps that have gone to fixation, or near-fixation, in one population
— where within-population iHS loses power because there's little
polymorphism left to compare — by using the second, still-polymorphic
cohort as a diversity reference.

**Diagram opportunity:** A comparison table/diagram summarising iHS vs
XP-EHH vs H12/G123 together: number of cohorts compared (one vs two),
data type required (phased haplotypes vs unphased diplotypes), and which
part of a sweep's lifecycle each is most sensitive to (ongoing/partial for
iHS, complete-in-one-population for XP-EHH, complete/near-complete for
H12/G123). Would fit well right here, drawing the whole notebook's
statistics together.

Parameters:
- **contig**, **analysis**, **sample_sets**, **sample_query_options**: as
  elsewhere.
- **cohort1_query** / **cohort2_query**: pandas queries defining the two
  cohorts being compared (same role as H1X's `cohort1_query`/
  `cohort2_query`, though typed here as a plain `sample_query` string).
- **window_size** / **percentiles**: as in `ihs_gwss`, summarise per-SNP
  XP-EHH into per-window percentiles.
- **filter_min_maf**: as in iHS, but MAF filtering here is based on the
  *combined* allele counts of both cohorts.
- **map_pos**: optional genetic-map (cM) position array to use instead of
  assuming uniform recombination when integrating EHH.
- **min_ehh**, **max_gap**, **gap_scale**, **include_edges**,
  **use_threads**: same meaning as the equivalent `ihs_gwss` parameters.
- **min_cohort_size** / **max_cohort_size**, **random_seed**, **chunks**,
  **inline_array**: as elsewhere.


In [17]:
x, xpehh = ag3.xpehh_gwss(
    contig="X",
    analysis="gamb_colu",
    window_size=1_000,
    cohort1_query=coh1_query,
    cohort2_query=mayotte_query,
    sample_sets="3.0",
    max_cohort_size=10,
)
df_xpehh = pd.DataFrame(xpehh, columns=[f"p{p}" for p in (50, 75, 100)])
df_xpehh.insert(0, "position", x)
df_xpehh.sort_values("p100", ascending=False).head()


,position,p50,p75,p100
73,1.512855e+07,2.192965,3.118653,4.398807
74,1.544523e+07,1.804230,2.351389,2.954673
100,2.403588e+07,-0.033375,0.375641,2.806968
95,2.020835e+07,-0.505610,-0.218347,2.753451
98,2.250200e+07,0.104707,0.434032,2.677320


## `plot_xpehh_gwss`

Runs `xpehh_gwss` and produces the two-panel (XP-EHH scatter + genes)
bokeh figure, with one colour-shaded series per percentile as in
`plot_ihs_gwss`.

Parameters: **palette**, plus the same layout parameters as
`plot_ihs_gwss`/`plot_h12_gwss` (**title**, **sizing_mode**, **width**,
**track_height**, **genes_height**, **show**, **output_backend**,
**gene_labels**, **gene_labelset**), applied to the XP-EHH track.


In [18]:
ag3.plot_xpehh_gwss(
    contig="X",
    analysis="gamb_colu",
    window_size=1_000,
    percentiles=(50, 60, 90, 100),
    cohort1_query=coh1_query,
    cohort2_query=mayotte_query,
    sample_sets="3.0",
    max_cohort_size=10,
)


Load genome features: ⠋ (0:00:00.00)

/Users/katie.barr/malariagen-data-python/malariagen_data/anoph/xpehh.py:473: UserWarning: found multiple competing values for 'toolbar.active_inspect' property; using the latest value
  fig = bokeh.layouts.gridplot(
